In [15]:
import torch.nn as nn

class SimpleFeatureExtractor(nn.Module):
    def __init__(self):
        super(SimpleFeatureExtractor, self).__init__()
        self.features = nn.Sequential(
            # Input: [Batch, 3, 224, 224]
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), 
            # Shape is now [Batch, 64, 112, 112]
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), 
            # Shape is now [Batch, 128, 56, 56]
            
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), 
            # Shape is now [Batch, 256, 28, 28]
            
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), 
            # Shape is now [Batch, 512, 14, 14]
            
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  
            # Final shape: [Batch, 512, 7, 7]
        )

    def forward(self, x):
        return self.features(x)

In [28]:
class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, 
                                   padding=1, stride=stride, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(out_channels) # Highly recommended for student models

    def forward(self, x):
        x = self.relu(self.depthwise(x))
        x = self.relu(self.bn(self.pointwise(x)))
        return x

class MobileStudentExtractor(nn.Module):
    def __init__(self):
        super(MobileStudentExtractor, self).__init__()
        
        # Initial thin convolution to map 3 channels to 32
        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # 224 -> 112
        )

        self.features = nn.Sequential(
            # Block 1: 112 -> 56
            DepthwiseSeparableBlock(32, 64),
            nn.MaxPool2d(2),
            
            # Block 2: 56 -> 28
            DepthwiseSeparableBlock(64, 128),
            nn.MaxPool2d(2),
            
            # Block 3: 28 -> 14
            DepthwiseSeparableBlock(128, 256),
            nn.MaxPool2d(2),
            
            # Block 4: 14 -> 7
            DepthwiseSeparableBlock(256, 512),
            nn.MaxPool2d(2),
            
            # Final Layer: Keep 512 channels
            DepthwiseSeparableBlock(512, 512)
        )

    def forward(self, x):
        x = self.initial(x)
        return self.features(x)

In [29]:
import torch
from torchvision.io import decode_image
from torchvision.models import vgg16, VGG16_Weights, vgg11, VGG11_Weights
import copy
from pet_dataset import PetDataset

# 1. Load the model and the pre-trained weights
weights = VGG11_Weights.DEFAULT
model = vgg11(weights=weights)

teacher = copy.deepcopy(model.features)
student = MobileStudentExtractor()

#dataset = PetDataset(
#        root_path="/home/gabriel/Documentos/Datasets/images",
#        transform=weights.transforms()
#    )

dataset = PetDataset(
        root_path="/home/gabriel/Documentos/Datasets/Oxford-Pet-Dataset/images",
        transform=weights.transforms()
    )

In [36]:
from train_student import train_student

train_student(student, teacher, dataset, 100, 0.0002)

Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


Epoch [1/100] | Avg Loss: 1.5648


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


Epoch [2/100] | Avg Loss: 1.5373


Corrupt JPEG data: 240 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


...

Epoch [98/100] | Avg Loss: 0.9891


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


Epoch [99/100] | Avg Loss: 0.9885


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


Epoch [100/100] | Avg Loss: 0.9859

Training complete!


In [18]:
class Rest_of_VGG(nn.Module):
    def __init__(self, num_classes=37): 
        super(Rest_of_VGG, self).__init__()
        weights = VGG11_Weights.DEFAULT

        model = vgg11(weights=weights)

        self.avgpool = model.avgpool
        self.classifier = model.classifier

        in_features = self.classifier[6].in_features
        self.classifier[6] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)
    

In [19]:
from train import train_classification

head = Rest_of_VGG()
train_classification(teacher, head, dataset, 10, 0.0001)

Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


Epoch 1/10 - Loss: 0.8377


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: 240 extraneous bytes before marker 0xd9


Epoch 2/10 - Loss: 0.2336


Corrupt JPEG data: 240 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


...

Epoch 8/10 - Loss: 0.0350


Corrupt JPEG data: 240 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


Epoch 9/10 - Loss: 0.0276


Corrupt JPEG data: 240 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


Epoch 10/10 - Loss: 0.0461


In [20]:
from train_student import evaluate_student


evaluate_student(teacher, head, dataset)

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


Accuracy: 99.73%


99.72936400541272

In [37]:
evaluate_student(student, head, dataset)

Corrupt JPEG data: 240 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


Accuracy: 87.96%


87.95669824086603

In [27]:
print("Numero de parametros do teacher: ", str(sum(p.numel() for p in teacher.parameters())))

Numero de parametros do teacher:  9220480


In [38]:
print("Numero de parametros do student: ", str(sum(p.numel() for p in student.parameters())))

Numero de parametros do student:  451456
